# Caracterización de las siete series con catch22

Este cuaderno cierra el Laboratorio 2 con el ejercicio de catch22. Extrae las 22 características canónicas de las siete series mensuales construidas en el Laboratorio 1, arma la matriz serie por característica y la analiza con PCA, clustering, mapa de calor, matriz de correlaciones y mapa de distancias entre series.

A diferencia de los cuadernos 09 y 10, que modelaron solo total y vía aérea sobre el conjunto de entrenamiento, aquí entran las siete series completas, de enero de 2009 a junio de 2026 (210 meses): el ejercicio es descriptivo y no de pronóstico, así que no hay partición que respetar. Produce `resultados/catch22_*.csv` y las figuras `catch22_*.png`.

## 1. La idea detrás de catch22

Comparar series por su dinámica obliga a elegir indicadores. En el cuaderno 07 los elegimos a mano: fuerza estacional, fuerza de tendencia, pendiente prepandemia, coeficiente de variación e impacto de la pandemia. Son cinco decisiones defendibles, pero arbitrarias, y nada garantiza que sean las que mejor separan estas siete series. La biblioteca `hctsa` lleva la idea al extremo opuesto y calcula miles de operaciones sobre una misma serie: exhaustivo, caro y muy redundante, porque cientos de esas operaciones miden casi lo mismo.

catch22 es el punto medio. Lubba et al. (2019) partieron de una versión filtrada de `hctsa` con 4,791 características y las evaluaron sobre 93 conjuntos de clasificación de series de tiempo, más de 147,000 series en total. Descartaron las que no superan al azar, agruparon las restantes por la similitud de su desempeño entre conjuntos —dos características que aciertan y fallan en los mismos problemas son redundantes— y conservaron un representante por grupo. De 4,791 quedaron 22. La reducción cuesta en promedio 7 % de exactitud de clasificación y devuelve un factor cercano a 1000 en tiempo de cómputo, con escalamiento casi lineal en la longitud de la serie.

Las 22 características cubren ocho familias: forma de la distribución de valores, ubicación de los eventos extremos, autocorrelación lineal, autocorrelación no lineal, contenido espectral y periodicidad, diferencias sucesivas y error de pronósticos locales, dinámica simbólica y rachas, y escalamiento de fluctuaciones. El nombre de cada una codifica la operación y sus parámetros: `CO_f1ecac` es el primer cruce de la autocorrelación por 1/e y `SB_BinaryStats_mean_longstretch1` es la racha más larga por encima de la media. La celda siguiente imprime el catálogo completo.

Un detalle que condiciona todo el ejercicio: las características se calculan sobre la serie estandarizada, así que describen forma y dinámica, no nivel ni escala. Por eso la variante `catch24` reincorpora la media y la desviación como dos características extra. Aquí se usan las 22 canónicas, que es lo que pide el enunciado.

La importancia práctica es que catch22 convierte una serie de longitud arbitraria en un vector de longitud fija e interpretable. Eso habilita el resto del ejercicio, PCA, clustering y distancias entre series, con herramienta multivariada ordinaria, y pone en el mismo plano a la serie total, con una media de 248,990 viajeros mensuales, y a vía marítima, con 5,851: la invariancia de escala evita que la magnitud domine la comparación, que es justo el problema que tuvo el comparativo del Laboratorio 1, donde cada indicador hubo que normalizarlo a mano. Y a diferencia de un vector aprendido por una red, cada coordenada tiene nombre y significado, de modo que las diferencias entre series se pueden explicar y no solo medir.

Queda una limitación declarada desde ahora: catch22 se seleccionó para clasificar series de benchmark y varias de sus características necesitan series largas. Las nuestras tienen 210 observaciones mensuales, así que las dos de escalamiento de fluctuaciones, que ajustan pendientes sobre varias escalas temporales, y las que dependen de la matriz de transición son las más expuestas a resultar inestables o constantes. El inciso 2 lo verifica antes de usarlas.

> Lubba, C. H., Sethi, S. S., Knaute, P., Schultz, S. R., Fulcher, B. D. y Jones, N. S. (2019). catch22: CAnonical Time-series CHaracteristics. *Data Mining and Knowledge Discovery*, 33(6), 1821-1852. arXiv:1901.10200.

La implementación usada es `pycatch22`, el binding oficial de la versión en C de los autores, agregado a `requirements-lab2.txt`.

In [1]:
from importlib.metadata import version
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.catch22 import catalogo

print(f"pycatch22 {version('pycatch22')}")

tabla = catalogo()
print(f"{len(tabla)} características en {tabla['familia'].nunique()} familias")
for familia, grupo in tabla.groupby("familia", sort=False):
    print(f"\n{familia}")
    for _, fila in grupo.iterrows():
        print(f"  {fila['caracteristica']:<44s}{fila['descripcion']}")

pycatch22 0.4.5
22 características en 8 familias

Distribución de valores
  DN_HistogramMode_5                          Moda de la distribución de valores, histograma de 5 bins
  DN_HistogramMode_10                         Moda de la distribución de valores, histograma de 10 bins

Autocorrelación lineal
  CO_f1ecac                                   Primer cruce de la ACF por 1/e
  CO_FirstMin_ac                              Retardo del primer mínimo de la ACF

Autocorrelación no lineal
  CO_HistogramAMI_even_2_5                    Información mutua con retardo 2, histograma de 5 bins
  CO_trev_1_num                               Asimetría temporal de las diferencias sucesivas (trev)
  CO_Embed2_Dist_tau_d_expfit_meandiff        Ajuste exponencial a las distancias en el espacio embebido 2-D
  IN_AutoMutualInfoStats_40_gaussian_fmmi     Primer mínimo de la información mutua, estimador gaussiano

Diferencias y pronóstico local
  MD_hrv_classic_pnn40                        Proporción de di

## 2. Extracción de las 22 características

Las características se calculan sobre la serie completa, los 210 meses de enero de 2009 a junio de 2026, y no sobre el conjunto de entrenamiento. Los cuadernos 09 y 10 respetaron la partición porque pronosticaban; aquí el objetivo es describir y comparar la dinámica de las siete series, no predecir, así que apartar 63 meses solo desperdiciaría información. Las series por país de residencia y vía marítima tienen meses en cero, y por eso quedaron fuera del modelado LSTM, que dependía de `log1p`; catch22 no necesita esa transformación, así que las siete entran completas.

`extraer_serie` llama a `pycatch22.catch22_all` y devuelve las 22 características indexadas por nombre, después de comprobar que la biblioteca las entregó en el orden del catálogo. La comprobación no es adorno: el catálogo del inciso 1 asigna familia y descripción por nombre, de modo que un reordenamiento en una versión distinta de `pycatch22` dejaría la matriz mal etiquetada sin que nada fallara.

La celda siguiente verifica además tres cosas antes de que el resto del cuaderno dependa de ellas.

- **Invariancia de escala.** `catch22_all` estandariza la serie internamente, así que `2 · serie + 1000` devuelve exactamente los mismos 22 valores. Eso es lo que hace comparable a la serie total, con media de 248,990 viajeros mensuales, con vía marítima, con 5,851, y lo que implica que ni el nivel ni la dispersión entran en la matriz: lo que queda es forma y dinámica.
- **Ausencia de valores faltantes.** Ninguna de las 154 celdas resulta NaN, ni en las series con meses en cero.
- **Variabilidad entre series.** Ninguna de las 22 características toma el mismo valor en las siete series. Esto matiza la limitación anunciada en el inciso 1: incluso las dos de escalamiento de fluctuaciones, que son las que más longitud exigen, discriminan entre series. Las 22 se conservan, y el inciso 4 podrá estandarizarlas por columna sin divisiones entre cero.

In [2]:
import numpy as np
import pandas as pd

from src.catch22 import extraer_serie
from src.utils import SERIES, cargar_serie

completas = {clave: cargar_serie(clave, "completa") for clave in SERIES}
extraidas = pd.DataFrame(
    {clave: extraer_serie(serie) for clave, serie in completas.items()}
).T

assert extraidas.shape == (len(SERIES), len(tabla))
assert np.allclose(extraidas.loc["total"], extraer_serie(2 * completas["total"] + 1000))

meses = sorted({len(serie) for serie in completas.values()})
constantes = list(extraidas.columns[extraidas.std(ddof=0) == 0])
print(f"series: {extraidas.shape[0]}, características: {extraidas.shape[1]}")
print(f"meses por serie: {meses}")
print(f"NaN en la extracción: {int(extraidas.isna().sum().sum())}")
print(f"características constantes entre series: {constantes or 'ninguna'}")
print("invariancia de escala: 2 · serie + 1000 devuelve los mismos 22 valores")

catalogo_indexado = tabla.set_index("caracteristica")
ejemplo = pd.DataFrame(
    {
        "valor": extraidas.loc["total"].round(4),
        "familia": catalogo_indexado["familia"],
        "descripcion": catalogo_indexado["descripcion"],
    }
)
print(f"\nvector de la serie total\n{ejemplo.to_string()}")

series: 7, características: 22
meses por serie: [210]
NaN en la extracción: 0
características constantes entre series: ninguna
invariancia de escala: 2 · serie + 1000 devuelve los mismos 22 valores

vector de la serie total
                                               valor                         familia                                                          descripcion
DN_HistogramMode_5                            0.1885         Distribución de valores             Moda de la distribución de valores, histograma de 5 bins
DN_HistogramMode_10                          -0.5804         Distribución de valores            Moda de la distribución de valores, histograma de 10 bins
CO_f1ecac                                     8.5789          Autocorrelación lineal                                       Primer cruce de la ACF por 1/e
CO_FirstMin_ac                               10.0000          Autocorrelación lineal                                  Retardo del primer mínimo de la ACF
CO_His

## 3. Matriz serie por característica

La matriz lleva las siete series en las filas y las 22 características en las columnas, que es la forma que pide el enunciado y también la que espera `scikit-learn`: una observación por fila. `matriz_caracteristicas` devuelve solo el bloque numérico, indexado por la clave de la serie. La etiqueta y la categoría se agregan al escribir `resultados/catch22_caracteristicas.csv`, con el mismo formato de `comparativo_series.csv` del Laboratorio 1, y las categorías se reutilizan de `src/comparativo.py` en lugar de redefinirlas, para que el inciso 10, que pregunta si las series de una misma categoría se agrupan, use exactamente la clasificación del comparativo anterior.

La matriz se imprime transpuesta, con las características en las filas, porque 22 columnas no caben legibles a lo ancho; es el mismo recurso que usa el cuaderno 07 para los perfiles estacionales.

Así impresa se ve el problema que resuelve el inciso 4: las columnas viven en escalas incomparables. `SB_BinaryStats_mean_longstretch1` va de 9 a 50 meses y `PD_PeriodicityWang_th0_01` de 2 a 11, mientras `SB_TransitionMatrix_3ac_sumdiagcov` se mueve entre 0.006 y 0.111. Una distancia euclidiana sobre la matriz cruda quedaría decidida por dos o tres columnas y las diecinueve restantes no aportarían nada.

In [3]:
from src.catch22 import matriz_caracteristicas
from src.comparativo import CATEGORIAS
from src.utils import RUTA_RESULTADOS

matriz = matriz_caracteristicas(completas)

assert list(matriz.index) == list(SERIES)
assert list(matriz.columns) == list(tabla["caracteristica"])

exportable = matriz.reset_index()
exportable.insert(1, "etiqueta", [SERIES[clave] for clave in matriz.index])
exportable.insert(2, "categoria", [CATEGORIAS[clave] for clave in matriz.index])
exportable.to_csv(RUTA_RESULTADOS / "catch22_caracteristicas.csv", index=False)

print(f"matriz {matriz.shape[0]} x {matriz.shape[1]} en resultados/catch22_caracteristicas.csv")
print(matriz.T.round(3).to_string())

rangos = (matriz.max() - matriz.min()).sort_values(ascending=False)
print("\nrecorrido de cada característica entre las siete series")
print(rangos.round(3).to_string())

matriz 7 x 22 en resultados/catch22_caracteristicas.csv
clave                                         total  via_aerea  via_terrestre  via_maritima  pais_el_salvador  pais_estados_unidos  pais_honduras
DN_HistogramMode_5                            0.189     -0.408         -0.547        -0.432            -0.367               -0.629         -0.222
DN_HistogramMode_10                          -0.580     -0.123         -0.782        -0.652            -0.590               -0.373         -0.432
CO_f1ecac                                     8.579      6.437          8.997         2.771            13.874                9.224         18.925
CO_FirstMin_ac                               10.000     10.000          3.000         6.000             3.000                2.000          8.000
CO_HistogramAMI_even_2_5                      0.454      0.207          0.537         0.123             0.476                0.229          0.477
CO_trev_1_num                                -0.127     -0.175      

## 4. Estandarización de las características

En este ejercicio conviven dos estandarizaciones que actúan en direcciones distintas y conviene no confundirlas.

| | Qué estandariza | Quién la aplica | Para qué |
|---|---|---|---|
| Interna de catch22 | Cada serie, sobre sus 210 meses | `pycatch22`, verificado en el inciso 2 | Que el nivel y la escala de la serie no entren en las características |
| La de este inciso | Cada característica, sobre las siete series | `StandardScaler` | Que ninguna columna domine distancias, PCA ni clustering |

La segunda es indispensable por lo que muestra el recorrido impreso en el inciso 3: `SB_BinaryStats_mean_longstretch1` varía 41 meses entre series y `CO_f1ecac` varía 16, mientras `SB_TransitionMatrix_3ac_sumdiagcov` varía 0.106. Sobre la matriz cruda, esas dos columnas decidirían casi solas cualquier distancia euclidiana y las otras veinte no aportarían nada. Después de estandarizar, las 22 entran con el mismo peso a priori. Se usa `StandardScaler` de `scikit-learn`, la misma biblioteca con la que el Laboratorio 2 escaló las series para la LSTM, con su convención de dividir entre la desviación poblacional (`ddof=0`).

Hay una consecuencia estadística que conviene declarar antes de interpretar el inciso 5: con siete observaciones por columna, el z-score de mayor magnitud posible es √6 ≈ 2.449. En las características donde una sola serie se despega del resto, como las dos de escalamiento de fluctuaciones, esa serie queda cerca del tope y las otras seis comprimidas en un rango estrecho de z positivos. No invalida el análisis, pero explica por qué esas columnas van a pesar tanto en las primeras componentes y en el mapa de distancias.

Un último punto que evita una confusión frecuente: la matriz de correlaciones entre características del inciso 5 sale idéntica sobre la matriz cruda o sobre la estandarizada, porque la correlación de Pearson es invariante a transformaciones afines por columna. La estandarización cambia el PCA, el clustering, las distancias y el mapa de calor; esa correlación, no.

In [4]:
from src.catch22 import estandarizar

estandarizada = estandarizar(matriz)

assert np.allclose(estandarizada.mean(), 0)
assert np.allclose(estandarizada.std(ddof=0), 1)

exportable_z = estandarizada.reset_index()
exportable_z.insert(1, "etiqueta", [SERIES[clave] for clave in estandarizada.index])
exportable_z.insert(2, "categoria", [CATEGORIAS[clave] for clave in estandarizada.index])
exportable_z.to_csv(RUTA_RESULTADOS / "catch22_estandarizado.csv", index=False)

print("cada característica queda con media 0 y desviación 1")
print(f"z de mayor magnitud posible con {len(estandarizada)} series: {np.sqrt(len(estandarizada) - 1):.3f}")
print(estandarizada.T.round(3).to_string())

print("\ncaracterística más extrema de cada serie")
for clave, fila in estandarizada.iterrows():
    extrema = fila.abs().idxmax()
    print(f"  {SERIES[clave]:<15s} {extrema:<44s} z = {fila[extrema]:+.2f}")

cada característica queda con media 0 y desviación 1
z de mayor magnitud posible con 7 series: 2.449
clave                                        total  via_aerea  via_terrestre  via_maritima  pais_el_salvador  pais_estados_unidos  pais_honduras
DN_HistogramMode_5                           2.146     -0.251         -0.811        -0.350            -0.089               -1.140          0.495
DN_HistogramMode_10                         -0.379      1.910         -1.385        -0.738            -0.429                0.656          0.364
CO_f1ecac                                   -0.259     -0.703         -0.173        -1.463             0.838               -0.125          1.885
CO_FirstMin_ac                               1.265      1.265         -0.949         0.000            -0.949               -1.265          0.632
CO_HistogramAMI_even_2_5                     0.626     -0.981          1.175        -1.532             0.776               -0.842          0.779
CO_trev_1_num                

## 5. Análisis sobre la matriz estandarizada

Los cinco análisis que pide el enunciado parten de la misma matriz 7 × 22 estandarizada del inciso 4. Antes de entrar, la restricción que atraviesa todo el inciso: hay **siete observaciones**. Eso acota el PCA a seis componentes, deja cualquier prueba de significancia sin poder y hace que un solo valor extremo mueva visiblemente los resultados. El análisis es descriptivo, y así se presenta.

La interpretación de fondo, qué series se parecen, qué características discriminan, qué grupos hay y cuáles series son atípicas, corresponde a los incisos 7 al 13. Estas celdas producen y describen la evidencia.

### 5.1 Análisis de componentes principales

El PCA se calcula sobre la matriz estandarizada, lo que equivale a un PCA sobre la matriz de correlaciones: cada característica llega con la misma varianza inicial y ninguna domina por su escala. Se piden seis componentes porque la matriz centrada de siete series tiene rango seis; una séptima componente tendría varianza numéricamente nula.

La figura tiene dos paneles. El primero es la varianza explicada por componente con su acumulada. El segundo es el plano PC1-PC2 con las siete series y, como flechas, las ocho características de mayor contribución al plano, medida como la norma de sus cargas en PC1 y PC2. Las flechas son lo que hace del gráfico un biplot: una serie situada en la dirección de una flecha puntúa alto en esa característica, y eso es lo que permitirá responder el inciso 8 con evidencia y no por intuición.

Las coordenadas de las siete series quedan en `resultados/catch22_pca.csv` y las cargas de las tres primeras componentes, con la familia de cada característica, en `resultados/catch22_pca_cargas.csv`.

In [5]:
from src.catch22 import FAMILIAS, analizar_pca, figura_pca

coordenadas, cargas, varianza = analizar_pca(estandarizada)
figura_pca(coordenadas, cargas, varianza)

exportable_pca = coordenadas.reset_index()
exportable_pca.insert(1, "etiqueta", [SERIES[clave] for clave in coordenadas.index])
exportable_pca.insert(2, "categoria", [CATEGORIAS[clave] for clave in coordenadas.index])
exportable_pca.round(4).to_csv(RUTA_RESULTADOS / "catch22_pca.csv", index=False)

cargas_exportables = cargas[["pc1", "pc2", "pc3"]].copy()
cargas_exportables.insert(0, "familia", pd.Series(FAMILIAS))
cargas_exportables.index.name = "caracteristica"
cargas_exportables.round(4).to_csv(RUTA_RESULTADOS / "catch22_pca_cargas.csv")

print("varianza explicada")
print(
    pd.DataFrame(
        {"varianza_%": 100 * varianza, "acumulada_%": 100 * varianza.cumsum()}
    )
    .round(1)
    .to_string()
)

print("\ncoordenadas de las series")
print(coordenadas.round(3).to_string())

contribucion = np.hypot(cargas["pc1"], cargas["pc2"]).sort_values(ascending=False)
resumen_cargas = pd.DataFrame(
    {
        "contribucion": contribucion,
        "pc1": cargas["pc1"],
        "pc2": cargas["pc2"],
        "familia": pd.Series(FAMILIAS),
    }
).loc[contribucion.index]
print("\ncaracterísticas de mayor contribución al plano PC1-PC2")
print(resumen_cargas.head(8).round(3).to_string())

varianza explicada
     varianza_%  acumulada_%
pc1        39.2         39.2
pc2        24.3         63.4
pc3        15.8         79.3
pc4        13.1         92.4
pc5         4.9         97.2
pc6         2.8        100.0

coordenadas de las series
                       pc1    pc2    pc3    pc4    pc5    pc6
clave                                                        
total                0.017  2.647  1.093  2.038  1.571  0.605
via_aerea            2.166  1.244  3.347 -0.501 -1.308 -0.403
via_terrestre       -1.543  0.070 -1.768  2.161 -0.451 -1.354
via_maritima         5.615  0.385 -2.546 -1.027  0.071  0.221
pais_el_salvador    -3.023 -0.303 -1.002  0.334 -1.402  1.296
pais_estados_unidos  0.301 -5.216  1.140  0.142  0.741  0.042
pais_honduras       -3.534  1.172 -0.263 -3.147  0.778 -0.407

características de mayor contribución al plano PC1-PC2
                                             contribucion    pc1    pc2                         familia
SC_FluctAnal_2_dfa_50_1_2_logi_pr

### 5.2 Clustering

El agrupamiento corre sobre las 22 características estandarizadas, no sobre las coordenadas del PCA. Reducir a dos componentes y agrupar después descartaría el 36.6 % de la varianza antes de medir la primera distancia; el PCA sirve para ver el resultado, no para producirlo. Así el dendrograma y el mapa de distancias del inciso 5.5 describen el mismo espacio.

El método es jerárquico aglomerativo con enlace de Ward sobre distancia euclidiana. Con siete series lo informativo es el orden en que se fusionan y a qué distancia, no una partición fija, y Ward es el enlace coherente con la distancia que se reporta en 5.5. El número de grupos se elige por el coeficiente de silueta de las particiones de Ward para k de 2 a 5. Con siete puntos la silueta es una medida gruesa: sirve para ordenar candidatos, no para sostener que existe un número óptimo de grupos.

Como verificación se corre k-means con ese mismo k, `n_init=10` y semilla 42, y se compara contra Ward con el índice de Rand ajustado. Si los dos métodos devuelven la misma partición, el resultado no depende del algoritmo, que es justo lo que el inciso 9 necesita saber antes de hablar de grupos naturales. Se guarda además la silueta de cada serie por separado: una serie con silueta cercana a cero no pertenece con claridad a ningún grupo, y ese es el primer indicio cuantitativo para el inciso 11.

El corte del dendrograma se pinta en el umbral exacto que produce el k elegido, de modo que los colores del árbol son los grupos que se reportan en `resultados/catch22_clusters.csv`.

In [6]:
from sklearn.metrics import adjusted_rand_score

from src.catch22 import agrupar, figura_clusters

enlace, grupos, siluetas = agrupar(estandarizada)
figura_clusters(enlace, grupos, siluetas)

k = int(siluetas.idxmax())
rand = adjusted_rand_score(grupos["grupo_ward"], grupos["grupo_kmeans"])

exportable_grupos = grupos.reset_index()
exportable_grupos.insert(1, "etiqueta", [SERIES[clave] for clave in grupos.index])
exportable_grupos.insert(2, "categoria", [CATEGORIAS[clave] for clave in grupos.index])
exportable_grupos.round(4).to_csv(RUTA_RESULTADOS / "catch22_clusters.csv", index=False)

print("silueta media por número de grupos")
print(siluetas.round(4).to_string())
print(f"\nk elegido: {k}")
print(f"Rand ajustado entre Ward y k-means: {rand:.3f}")

print("\ngrupos de Ward")
for numero, bloque in grupos.groupby("grupo_ward"):
    integrantes = ", ".join(SERIES[clave] for clave in bloque.index)
    print(f"  grupo {numero}: {integrantes}")

print("\ngrupos de k-means")
for numero, bloque in grupos.groupby("grupo_kmeans"):
    integrantes = ", ".join(SERIES[clave] for clave in bloque.index)
    print(f"  grupo {numero}: {integrantes}")

print("\nsilueta por serie")
for clave, fila in grupos.sort_values("silueta").iterrows():
    print(
        f"  {SERIES[clave]:<15s} grupo {int(fila['grupo_ward'])}"
        f"  silueta {fila['silueta']:+.3f}"
    )

print("\naltura de las fusiones de Ward")
print(np.round(np.sort(enlace[:, 2]), 3))

silueta media por número de grupos
k
2    0.1534
3    0.1628
4    0.1609
5    0.1265

k elegido: 3
Rand ajustado entre Ward y k-means: 0.444

grupos de Ward
  grupo 1: Total, Vía Aérea, Vía Marítima
  grupo 2: Vía Terrestre, El Salvador, Honduras
  grupo 3: Estados Unidos

grupos de k-means
  grupo 1: Total, Vía Terrestre, El Salvador, Honduras
  grupo 2: Estados Unidos
  grupo 3: Vía Aérea, Vía Marítima

silueta por serie
  Total           grupo 1  silueta -0.102
  Estados Unidos  grupo 3  silueta +0.000
  Vía Marítima    grupo 1  silueta +0.140
  Vía Aérea       grupo 1  silueta +0.154
  Vía Terrestre   grupo 2  silueta +0.261
  Honduras        grupo 2  silueta +0.317
  El Salvador     grupo 2  silueta +0.371

altura de las fusiones de Ward
[3.766 5.236 5.982 8.071 8.268 9.854]


La silueta es casi plana entre k = 2, 3 y 4 (0.153, 0.163 y 0.161), así que el k = 3 elegido gana por 0.002 sobre k = 4 y no debe leerse como un óptimo nítido. Ward y k-means discrepan en una sola serie, la total: Ward la fusiona con vía aérea y vía marítima, y k-means la coloca con vía terrestre, El Salvador y Honduras. Es también la única serie con silueta negativa, −0.102, es decir la única que queda más cerca del grupo vecino que del propio. El índice de Rand ajustado cae a 0.444 por ese único cambio, penalización esperable con siete observaciones. Las otras seis series salen idénticas con los dos métodos, y Estados Unidos aparece aislado en ambos.

### 5.3 Mapa de calor de las características

La figura dibuja la matriz estandarizada completa. Las 22 características van en las filas, agrupadas por familia y separadas con líneas, y las siete series en las columnas, ordenadas según las hojas del dendrograma de 5.2 y no según `SERIES`, de modo que los bloques de color coincidan con los grupos ya reportados. Es la misma orientación con la que se imprimieron las matrices de los incisos 3 y 4, así que la figura y las tablas se leen igual.

La escala de color es divergente y está fijada en ±2.449, el z de mayor magnitud posible con siete series, en lugar de ajustarse al máximo observado. Con eso, un rojo intenso significa siempre lo mismo y el blanco es siempre el promedio de las siete series en esa característica.

No hay CSV nuevo: la matriz dibujada es exactamente `resultados/catch22_estandarizado.csv`.

In [7]:
from src.catch22 import figura_heatmap, orden_dendrograma

orden = orden_dendrograma(enlace, estandarizada.index)
figura_heatmap(estandarizada, orden)

print("orden de las columnas, el de las hojas del dendrograma")
print(" | ".join(SERIES[clave] for clave in orden))

extremas = estandarizada.stack()
extremas = extremas[extremas.abs() > 2].sort_values()
print("\nceldas con |z| > 2, las que dominan el mapa")
for (clave, caracteristica), valor in extremas.items():
    print(f"  {SERIES[clave]:<15s} {caracteristica:<44s} z = {valor:+.2f}")

orden de las columnas, el de las hojas del dendrograma
Vía Marítima | Total | Vía Aérea | Estados Unidos | Honduras | Vía Terrestre | El Salvador

celdas con |z| > 2, las que dominan el mapa
  Estados Unidos  SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1       z = -2.42
  Estados Unidos  SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1  z = -2.41
  Vía Marítima    MD_hrv_classic_pnn40                         z = -2.29
  Total           DN_HistogramMode_5                           z = +2.15
  Vía Marítima    FC_LocalSimple_mean1_tauresrat               z = +2.34
  Vía Marítima    SP_Summaries_welch_rect_centroid             z = +2.36
  Vía Marítima    DN_OutlierInclude_n_001_mdrmd                z = +2.41
  Honduras        SB_TransitionMatrix_3ac_sumdiagcov           z = +2.42


### 5.4 Matriz de correlaciones entre características

La matriz es la correlación de Pearson entre las 22 características, tomando las siete series como observaciones, en el mismo orden por familia del mapa de calor para que las dos figuras se lean juntas. Se calcula sobre la matriz estandarizada, aunque el resultado es idéntico al de la matriz cruda por la invariancia anotada en el inciso 4.

Aquí hace falta una advertencia, porque la figura invita a sobreinterpretar. Cada coeficiente proviene de **siete puntos**. Con n = 7 el umbral nominal de significancia al 5 % es |r| ≈ 0.754, y la matriz tiene 231 pares distintos, así que por puro azar cabe esperar más de diez pares que cruzarían ese umbral sin que exista ninguna relación. La matriz sirve para detectar redundancia gruesa entre características *en este conjunto concreto de siete series*, no para afirmar dependencias generales entre ellas.

Conviene distinguir esto del diseño de catch22. Sus autores minimizaron la redundancia por similitud de desempeño en clasificación sobre 93 conjuntos de datos, no por correlación lineal en siete series mensuales de turismo. Que dos características salgan casi colineales aquí no contradice el diseño: significa que en estas siete series, y solo en ellas, aportan la misma información.

In [8]:
from src.catch22 import correlaciones, figura_correlaciones

correlacion = correlaciones(estandarizada)
figura_correlaciones(correlacion)
correlacion.round(4).to_csv(RUTA_RESULTADOS / "catch22_correlaciones.csv")

triangulo = np.triu_indices(len(correlacion), k=1)
pares = pd.Series(
    correlacion.to_numpy()[triangulo],
    index=pd.MultiIndex.from_arrays(
        [correlacion.index[triangulo[0]], correlacion.columns[triangulo[1]]]
    ),
)

print(f"pares distintos: {len(pares)}")
print(f"correlación media en valor absoluto: {pares.abs().mean():.3f}")
print(f"pares con |r| >= 0.9: {int((pares.abs() >= 0.9).sum())}")
print(f"pares con |r| >= 0.95: {int((pares.abs() >= 0.95).sum())}")

print("\npares con |r| >= 0.95")
fuertes = pares[pares.abs() >= 0.95].sort_values(key=abs, ascending=False)
for (una, otra), valor in fuertes.items():
    print(f"  {valor:+.3f}  {una} / {otra}")

pares distintos: 231
correlación media en valor absoluto: 0.398
pares con |r| >= 0.9: 6
pares con |r| >= 0.95: 4

pares con |r| >= 0.95
  +0.997  FC_LocalSimple_mean1_tauresrat / SP_Summaries_welch_rect_centroid
  +0.980  SP_Summaries_welch_rect_centroid / DN_OutlierInclude_n_001_mdrmd
  +0.965  SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1 / SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1
  +0.964  FC_LocalSimple_mean1_tauresrat / DN_OutlierInclude_n_001_mdrmd


### 5.5 Mapa de distancias entre series

El último de los cinco análisis que pide el inciso 2.4 es la matriz de distancias euclidianas entre las siete series, calculada sobre la misma matriz estandarizada de 22 características que alimentó el PCA y el clustering. Es la contraparte numérica del dendrograma de 5.2: Ward decide con esta misma distancia qué series fusionar primero, y aquí se ve el valor exacto detrás de cada fusión, entre todos los pares y no solo entre los que Ward unió.

`distancias` aplica `scipy.spatial.distance.pdist` con métrica euclidiana sobre las filas de la matriz estandarizada y devuelve la matriz cuadrada de 7 × 7 con las claves de las series en filas y columnas. La diagonal es cero por construcción, la matriz es simétrica, y por eso `figura_distancias` solo necesita anotar el triángulo completo, sin ocultar ninguna mitad: a diferencia del heatmap y la matriz de correlaciones, aquí no hay signo que perder al plegar la matriz.

Las filas y columnas se ordenan igual que en el mapa de calor de 5.3, según las hojas del dendrograma de Ward, para que las tres figuras, dendrograma, mapa de calor y mapa de distancias, cuenten la misma historia con el mismo orden de series. La escala de color es secuencial y no divergente, porque una distancia euclidiana no tiene signo: cero es el extremo de similitud máxima, no un punto medio neutro entre dos direcciones.

La matriz completa queda en `resultados/catch22_distancias.csv`.

In [9]:
from src.catch22 import distancias, figura_distancias

distancia = distancias(estandarizada)
figura_distancias(distancia, orden)
distancia.round(4).to_csv(RUTA_RESULTADOS / "catch22_distancias.csv")

print("matriz de distancias euclidianas (orden del dendrograma)")
print(distancia.loc[orden, orden].rename(columns=SERIES, index=SERIES).round(2).to_string())

triangulo = np.triu_indices(len(distancia), k=1)
pares_dist = pd.Series(
    distancia.to_numpy()[triangulo],
    index=pd.MultiIndex.from_arrays(
        [distancia.index[triangulo[0]], distancia.columns[triangulo[1]]]
    ),
)

mas_cercano = pares_dist.idxmin()
mas_lejano = pares_dist.idxmax()
print(f"\npar más similar:  {SERIES[mas_cercano[0]]} / {SERIES[mas_cercano[1]]}  d = {pares_dist[mas_cercano]:.3f}")
print(f"par más distinto: {SERIES[mas_lejano[0]]} / {SERIES[mas_lejano[1]]}  d = {pares_dist[mas_lejano]:.3f}")

print("\ndistancia media de cada serie al resto (indicio de atipicidad)")
medias = (distancia.sum() / (len(distancia) - 1)).sort_values(ascending=False)
for clave, valor in medias.items():
    print(f"  {SERIES[clave]:<15s} {valor:.3f}")

matriz de distancias euclidianas (orden del dendrograma)
clave           Vía Marítima  Total  Vía Aérea  Estados Unidos  Honduras  Vía Terrestre  El Salvador
clave                                                                                               
Vía Marítima            0.00   7.84       7.07            8.66      9.74           8.05         9.09
Total                   7.84   0.00       5.24            8.16      6.72           5.02         5.88
Vía Aérea               7.07   5.24       0.00            7.41      7.54           7.07         7.20
Estados Unidos          8.66   8.16       7.41            0.00      8.28           6.87         6.78
Honduras                9.74   6.72       7.54            8.28      0.00           6.17         4.77
Vía Terrestre           8.05   5.02       7.07            6.87      6.17           0.00         3.77
El Salvador             9.09   5.88       7.20            6.78      4.77           3.77         0.00

par más similar:  Vía Terrestre /

## 6. Análisis e interpretación

Los incisos 5.1 a 5.5 produjeron la evidencia; los que siguen la interpretan. Antes de responder conviene consolidar en una sola tabla lo que cada análisis dijo de cada serie, porque las preguntas de los incisos 7 a 13 se responden cruzando esas piezas y no mirándolas por separado: el PCA ubica, el clustering agrupa, la matriz de distancias mide y el mapa de calor explica *por qué*.

La tabla siguiente reúne, para cada una de las siete series, siete piezas de evidencia ya calculadas:

| Columna | De dónde viene | Qué responde |
|---|---|---|
| `grupo_ward`, `grupo_kmeans` | Inciso 5.2 | A qué grupo pertenece y si el resultado depende del algoritmo |
| `silueta` | Inciso 5.2 | Qué tan bien pertenece a su grupo; cerca de 0 o negativa significa que no pertenece con claridad |
| `pc1`, `pc2` | Inciso 5.1 | Dónde queda en el plano principal, que explica el 63.4 % de la varianza |
| `distancia_media` | Inciso 5.5 | Qué tan lejos está del resto en promedio; el indicador directo de atipicidad |
| `vecino_mas_cercano`, `distancia_vecino` | Inciso 5.5 | Con qué serie se parece más y cuánto |
| `caracteristica_extrema`, `z_extremo` | Inciso 5.3 | Qué característica la separa más del promedio de las siete |

Una advertencia que atraviesa todos los incisos que siguen y que no se repetirá en cada uno. Hay **siete series**. Ningún resultado de esta sección tiene respaldo inferencial: no hay pruebas de hipótesis con poder, los intervalos de confianza serían inútiles y una sola serie distinta cambiaría varias de las conclusiones. Todo lo que sigue es descriptivo, y las afirmaciones se acompañan del número que las sostiene para que el lector juzgue qué tan firme es cada una.

La tabla queda en `resultados/catch22_resumen_series.csv`.

In [10]:
vecinos = {}
for clave in distancia.index:
    otras = distancia.loc[clave].drop(clave)
    vecinos[clave] = (otras.idxmin(), otras.min())

extremas_serie = {clave: fila.abs().idxmax() for clave, fila in estandarizada.iterrows()}

resumen = pd.DataFrame(
    {
        "etiqueta": [SERIES[clave] for clave in estandarizada.index],
        "categoria": [CATEGORIAS[clave] for clave in estandarizada.index],
        "grupo_ward": grupos["grupo_ward"],
        "grupo_kmeans": grupos["grupo_kmeans"],
        "silueta": grupos["silueta"],
        "pc1": coordenadas["pc1"],
        "pc2": coordenadas["pc2"],
        "distancia_media": distancia.sum() / (len(distancia) - 1),
        "vecino_mas_cercano": [SERIES[vecinos[clave][0]] for clave in estandarizada.index],
        "distancia_vecino": [vecinos[clave][1] for clave in estandarizada.index],
        "caracteristica_extrema": [extremas_serie[clave] for clave in estandarizada.index],
        "z_extremo": [
            estandarizada.loc[clave, extremas_serie[clave]] for clave in estandarizada.index
        ],
    },
    index=estandarizada.index,
)

resumen.round(4).to_csv(RUTA_RESULTADOS / "catch22_resumen_series.csv")

print("evidencia consolidada por serie")
print(
    resumen[
        [
            "etiqueta",
            "categoria",
            "grupo_ward",
            "grupo_kmeans",
            "silueta",
            "pc1",
            "pc2",
            "distancia_media",
        ]
    ]
    .round(3)
    .to_string()
)

print("\nvecino más cercano y característica más extrema de cada serie")
for clave, fila in resumen.iterrows():
    print(
        f"  {fila['etiqueta']:<15s} vecino {fila['vecino_mas_cercano']:<15s} d = {fila['distancia_vecino']:.2f}"
        f"   {fila['caracteristica_extrema']:<44s} z = {fila['z_extremo']:+.2f}"
    )

evidencia consolidada por serie
                           etiqueta           categoria  grupo_ward  grupo_kmeans  silueta    pc1    pc2  distancia_media
clave                                                                                                                    
total                         Total          Referencia           1             1   -0.102  0.017  2.647            6.475
via_aerea                 Vía Aérea      Vía de ingreso           1             3    0.154  2.166  1.244            6.921
via_terrestre         Vía Terrestre      Vía de ingreso           2             1    0.261 -1.543  0.070            6.159
via_maritima           Vía Marítima      Vía de ingreso           1             3    0.140  5.615  0.385            8.410
pais_el_salvador        El Salvador  País de residencia           2             1    0.371 -3.023 -0.303            6.247
pais_estados_unidos  Estados Unidos  País de residencia           3             2    0.000  0.301 -5.216          